
# Amazon Search Ranking — Week 6 V2 FIXED
## Leakage-Safe Zero-Shot Ranking Adapter

This is the corrected V2 notebook.

### Why this fixed version exists
The first V2 version used stable sorting when many zero-shot scores were tied. Because the V1 file happened to be stored in LambdaMART order, tied zero-shot scores could inherit that order.

This version fixes the issue by:

1. Using a **deterministic product-hash tie breaker** independent of row order.
2. Automatically dropping zero-shot features that almost never vary within a query.
3. Verifying that **shuffling input rows does not change the final metrics**.
4. Keeping the query-level 60% tuning / 40% held-out split.
5. Never using review/rating/history/LambdaMART/ListNet scores as adapter inputs.

### Input
Upload:

`amazon_week6_v1_coldstart_benchmark_results.zip`

### Output
`amazon_week6_v2_fixed_zero_shot_adapter_results.zip`


In [ ]:

# 0. SETUP
!pip -q install pyarrow

import os, re, json, hashlib, shutil, zipfile, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path("/content/week6_v2_fixed")
UPLOAD_DIR = ROOT / "uploads"
EXTRACT_DIR = ROOT / "extracted"
OUT_DIR = ROOT / "outputs"

for p in [ROOT, UPLOAD_DIR, EXTRACT_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TUNE_FRACTION = 0.60
N_RANDOM_WEIGHTS = 3000
RANDOM_SEED = 42
MIN_VARYING_QUERY_SHARE = 0.10

print("Week 6 V2 FIXED environment ready.")


In [ ]:

# 1. UPLOAD V1 ZIP
from google.colab import files

uploaded = files.upload()
for name, content in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(content)

zips = list(UPLOAD_DIR.glob("*.zip"))
if not zips:
    raise RuntimeError("Please upload amazon_week6_v1_coldstart_benchmark_results.zip")

for zpath in zips:
    dest = EXTRACT_DIR / zpath.stem
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(dest)

def find_file(name):
    hits = list(EXTRACT_DIR.rglob(name))
    return hits[0] if hits else None

cold_path = find_file("cold_start_subset.csv")
if cold_path is None:
    raise RuntimeError("cold_start_subset.csv not found in V1 ZIP.")

cold = pd.read_csv(cold_path)

print("Loaded:", cold.shape)
print("Queries:", cold["__query__"].nunique())
print("Products:", cold["__product__"].nunique())


In [ ]:

# 2. ZERO-SHOT FEATURE POLICY

PROHIBITED_KEYWORDS = [
    "review", "rating", "click", "purchase", "order", "sale", "interaction",
    "impression", "view_count", "popularity", "history", "historical",
    "lambdamart", "listnet", "ltr_score", "predicted_rank"
]

FEATURE_CANDIDATES = OrderedDict([
    ("BM25", ["bm25_score_qnorm", "bm25_score"]),
    ("TFIDF", ["tfidf_similarity_qnorm", "tfidf_similarity", "tfidf_similarity_diagnostic"]),
    ("SBERT", ["sbert_similarity_qnorm", "sbert_similarity"]),
    ("CLIP", ["clip_similarity_qnorm", "clip_similarity"]),
    ("TITLE_COVERAGE", ["query_title_token_coverage", "query_title_overlap"]),
    ("DESC_OVERLAP", ["query_description_overlap"]),
    ("EXACT_TITLE", ["exact_title_match"]),
    ("QUERY_IN_TITLE", ["query_in_title"]),
    ("BRAND_MATCH", ["brand_in_query"]),
    ("CATEGORY_MATCH", ["category_token_match"]),
])

def prohibited(c):
    x = str(c).lower()
    return any(k in x for k in PROHIBITED_KEYWORDS)

detected = OrderedDict()
for group, candidates in FEATURE_CANDIDATES.items():
    for c in candidates:
        if c in cold.columns and not prohibited(c):
            detected[group] = c
            break

if len(detected) < 2:
    raise RuntimeError("Too few zero-shot features detected.")

print("Detected candidate zero-shot features:")
for g, c in detected.items():
    print(f" - {g:16s} -> {c}")


In [ ]:

# 3. FEATURE DIAGNOSTICS + LOW-INFORMATION FILTER

if "__label__" not in cold.columns:
    raise RuntimeError("__label__ is missing.")

cold["__label__"] = pd.to_numeric(cold["__label__"], errors="coerce").fillna(0.0)

diag_rows = []
multi_query_count = sum(1 for _, g in cold.groupby("__query__") if len(g) >= 2)

for group, c in detected.items():
    vals = pd.to_numeric(cold[c], errors="coerce").fillna(0.0)
    temp = cold[["__query__"]].copy()
    temp["_v"] = vals

    varying = 0
    for _, g in temp.groupby("__query__"):
        if len(g) < 2:
            continue
        if g["_v"].nunique(dropna=False) > 1:
            varying += 1

    share = varying / max(multi_query_count, 1)
    diag_rows.append({
        "feature_group": group,
        "source_column": c,
        "n_unique_global": int(vals.nunique()),
        "nonzero_share": float((vals != 0).mean()),
        "varying_multiquery_count": varying,
        "varying_multiquery_share": share,
        "keep_for_adapter": share >= MIN_VARYING_QUERY_SHARE,
    })

feature_diag = pd.DataFrame(diag_rows)

print("Feature diagnostics:")
display(feature_diag)

selected = OrderedDict()
for _, r in feature_diag[feature_diag["keep_for_adapter"]].iterrows():
    selected[r["feature_group"]] = r["source_column"]

if len(selected) < 2:
    raise RuntimeError("Too few discriminative zero-shot features remain after filtering.")

print("\nSelected adapter features:")
for g, c in selected.items():
    print(f" - {g:16s} -> {c}")


In [ ]:

# 4. QUERY NORMALIZATION

norm_cols = []
for group, c in selected.items():
    vals = pd.to_numeric(cold[c], errors="coerce").fillna(0.0)
    mn = vals.groupby(cold["__query__"]).transform("min")
    mx = vals.groupby(cold["__query__"]).transform("max")
    den = mx - mn

    out_c = f"zs_{group.lower()}"
    cold[out_c] = np.where(
        den.abs() > 1e-12,
        (vals - mn) / den,
        0.0
    )
    cold[out_c] = np.clip(cold[out_c].astype(float), 0.0, 1.0)
    norm_cols.append(out_c)

print(norm_cols)


In [ ]:

# 5. QUERY SPLIT + ORDER-INDEPENDENT TIE BREAKER

def stable_unit(value, salt=""):
    h = hashlib.sha1((salt + str(value)).encode("utf-8")).hexdigest()
    return int(h[:12], 16) / float(16**12 - 1)

cold["v2_split"] = cold["__query__"].map(
    lambda q: "tune" if stable_unit(q) < TUNE_FRACTION else "holdout"
)

# IMPORTANT:
# This tie breaker depends only on product ID, never on file row order,
# relevance labels, or any baseline model score.
cold["__tie_key__"] = cold["__product__"].map(
    lambda p: stable_unit(p, "week6_v2_tie:")
)

tune = cold[cold["v2_split"] == "tune"].copy()
holdout = cold[cold["v2_split"] == "holdout"].copy()

print("Tune queries:", tune["__query__"].nunique())
print("Holdout queries:", holdout["__query__"].nunique())


In [ ]:

# 6. LEAKAGE-SAFE METRICS

def safe_order(scores, tie_keys):
    scores = np.asarray(scores, dtype=float)
    tie_keys = np.asarray(tie_keys, dtype=float)
    # Primary key: descending score
    # Secondary key: deterministic product hash
    return np.lexsort((tie_keys, -scores))

def dcg_at_k(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    if len(rels) == 0:
        return np.nan
    return float(np.sum(
        ((2.0 ** rels) - 1.0) /
        np.log2(np.arange(2, len(rels) + 2))
    ))

def ndcg_at_k(y, s, tie, k):
    y = np.asarray(y, dtype=float)
    order = safe_order(s, tie)
    ideal = np.sort(y)[::-1]
    denom = dcg_at_k(ideal, k)
    if not np.isfinite(denom) or denom <= 0:
        return np.nan
    return dcg_at_k(y[order], k) / denom

def recall_at_k(y, s, tie, k):
    y = np.asarray(y, dtype=float)
    relevant = y > 0
    total = relevant.sum()
    if total == 0:
        return np.nan
    order = safe_order(s, tie)[:k]
    return float(relevant[order].sum() / total)

def reciprocal_rank(y, s, tie):
    y = np.asarray(y, dtype=float)
    order = safe_order(s, tie)
    hits = np.where(y[order] > 0)[0]
    return 0.0 if len(hits) == 0 else float(1.0 / (hits[0] + 1))

def evaluate(df, score_col=None, score_values=None, model="model", split="split"):
    work = df.copy()
    if score_values is not None:
        work["__eval_score__"] = np.asarray(score_values, dtype=float)
        sc = "__eval_score__"
    else:
        sc = score_col
        work[sc] = pd.to_numeric(work[sc], errors="coerce").fillna(0.0)

    rows = []
    for q, g in work.groupby("__query__", sort=False):
        if len(g) < 2:
            continue

        y = g["__label__"].to_numpy(float)
        s = g[sc].to_numpy(float)
        t = g["__tie_key__"].to_numpy(float)

        rows.append({
            "split": split,
            "model": model,
            "query": q,
            "n_candidates": len(g),
            "ndcg@10": ndcg_at_k(y, s, t, 10),
            "ndcg@3": ndcg_at_k(y, s, t, 3),
            "recall@10": recall_at_k(y, s, t, 10),
            "recall@3": recall_at_k(y, s, t, 3),
            "mrr": reciprocal_rank(y, s, t),
        })

    pq = pd.DataFrame(rows)

    summary = {
        "split": split,
        "model": model,
        "n_queries": int(len(pq)),
        "ndcg@10": float(pq["ndcg@10"].mean()) if len(pq) else np.nan,
        "ndcg@3": float(pq["ndcg@3"].mean()) if len(pq) else np.nan,
        "recall@10": float(pq["recall@10"].mean()) if len(pq) else np.nan,
        "recall@3": float(pq["recall@3"].mean()) if len(pq) else np.nan,
        "mrr": float(pq["mrr"].mean()) if len(pq) else np.nan,
    }
    return summary, pq


In [ ]:

# 7. FAST TUNING ARRAYS

def build_arrays(df):
    arrays = []
    for q, g in df.groupby("__query__", sort=False):
        if len(g) < 2:
            continue
        arrays.append({
            "X": g[norm_cols].to_numpy(float),
            "y": g["__label__"].to_numpy(float),
            "tie": g["__tie_key__"].to_numpy(float),
        })
    return arrays

tune_arrays = build_arrays(tune)

def fast_metrics(arrays, w):
    nd10, nd3, r3, mrr = [], [], [], []

    for item in arrays:
        y = item["y"]
        s = item["X"] @ w
        t = item["tie"]

        nd10.append(ndcg_at_k(y, s, t, 10))
        nd3.append(ndcg_at_k(y, s, t, 3))
        r3.append(recall_at_k(y, s, t, 3))
        mrr.append(reciprocal_rank(y, s, t))

    return {
        "ndcg@10": float(np.nanmean(nd10)),
        "ndcg@3": float(np.nanmean(nd3)),
        "recall@3": float(np.nanmean(r3)),
        "mrr": float(np.nanmean(mrr)),
    }

def objective(m):
    return 0.55*m["ndcg@10"] + 0.25*m["ndcg@3"] + 0.20*m["mrr"]

print("Tuning ranking queries:", len(tune_arrays))


In [ ]:

# 8. WEIGHT SEARCH — TUNING SET ONLY

rng = np.random.default_rng(RANDOM_SEED)
groups = list(selected.keys())
n = len(groups)

candidates = []

# Single feature anchors
for j in range(n):
    w = np.zeros(n)
    w[j] = 1.0
    candidates.append(("single", w))

# Equal weights
candidates.append(("equal", np.ones(n) / n))

# Hand seeds
def add_seed(weight_map):
    w = np.zeros(n)
    for g, val in weight_map.items():
        if g in groups:
            w[groups.index(g)] = val
    if w.sum() > 0:
        candidates.append(("seed", w / w.sum()))

add_seed({"BM25": .35, "SBERT": .45, "CLIP": .10, "TITLE_COVERAGE": .10})
add_seed({"BM25": .20, "TFIDF": .10, "SBERT": .50, "TITLE_COVERAGE": .20})
add_seed({"SBERT": .55, "TITLE_COVERAGE": .25, "DESC_OVERLAP": .10, "BM25": .10})

# Random simplex candidates
for _ in range(N_RANDOM_WEIGHTS):
    candidates.append(("random", rng.dirichlet(np.ones(n) * 0.8)))

records = []
best = None

for source, w in candidates:
    m = fast_metrics(tune_arrays, w)
    obj = objective(m)

    row = {"source": source, "objective": obj, **m}
    for g, weight in zip(groups, w):
        row[f"w_{g.lower()}"] = float(weight)
    records.append(row)

    if best is None or obj > best["objective"]:
        best = {
            "objective": obj,
            "weights": w.copy(),
            "metrics": m,
            "source": source,
        }

search_results = pd.DataFrame(records).sort_values(
    "objective", ascending=False
).reset_index(drop=True)

best_w = best["weights"]

print("Best tuning metrics:", best["metrics"])
print("\nBest weights:")
for g, w in zip(groups, best_w):
    print(f" - {g:16s}: {w:.4f}")

display(search_results.head(15))


In [ ]:

# 9. HELD-OUT EVALUATION

cold["zero_shot_adapter_score"] = cold[norm_cols].to_numpy(float) @ best_w
tune = cold[cold["v2_split"] == "tune"].copy()
holdout = cold[cold["v2_split"] == "holdout"].copy()

baseline_candidates = OrderedDict([
    ("BM25", "bm25_score_qnorm"),
    ("SBERT", "sbert_similarity_qnorm"),
    ("CLIP", "clip_similarity_qnorm"),
    ("LambdaMART", "lambdamart_score"),
    ("ListNet", "listnet_score"),
])

baseline_candidates = OrderedDict(
    (m, c) for m, c in baseline_candidates.items()
    if c in cold.columns
)

summaries = []
per_query_parts = []

for split_name, part in [
    ("tune", tune),
    ("holdout", holdout),
    ("full_cold_start", cold),
]:
    for model, col in baseline_candidates.items():
        s, pq = evaluate(part, score_col=col, model=model, split=split_name)
        summaries.append(s)
        per_query_parts.append(pq)

    s, pq = evaluate(
        part,
        score_col="zero_shot_adapter_score",
        model="ZeroShotAdapter",
        split=split_name,
    )
    summaries.append(s)
    per_query_parts.append(pq)

metrics = pd.DataFrame(summaries)
per_query = pd.concat(per_query_parts, ignore_index=True)

holdout_metrics = metrics[metrics["split"] == "holdout"].copy()
full_metrics = metrics[metrics["split"] == "full_cold_start"].copy()

def add_improvement(d):
    d = d.copy()
    d["ndcg10_improvement_vs_bm25_pct"] = np.nan
    d["ndcg3_improvement_vs_bm25_pct"] = np.nan
    d["mrr_improvement_vs_bm25_pct"] = np.nan

    if "BM25" in set(d["model"]):
        base = d[d["model"] == "BM25"].iloc[0]
        for metric, out_col in [
            ("ndcg@10", "ndcg10_improvement_vs_bm25_pct"),
            ("ndcg@3", "ndcg3_improvement_vs_bm25_pct"),
            ("mrr", "mrr_improvement_vs_bm25_pct"),
        ]:
            if base[metric] != 0:
                d[out_col] = (d[metric] - base[metric]) / base[metric] * 100
    return d

holdout_metrics = add_improvement(holdout_metrics)
full_metrics = add_improvement(full_metrics)

print("OFFICIAL HELD-OUT V2 RESULTS")
display(holdout_metrics.sort_values("ndcg@10", ascending=False))

print("\nFULL COLD-START DIAGNOSTIC")
display(full_metrics.sort_values("ndcg@10", ascending=False))


In [ ]:

# 10. ORDER-INDEPENDENCE REGRESSION TEST

# Metrics must be unchanged after arbitrary row shuffling.
shuffled = holdout.sample(frac=1.0, random_state=999).reset_index(drop=True)

orig_summary, _ = evaluate(
    holdout,
    score_col="zero_shot_adapter_score",
    model="ZeroShotAdapter",
    split="holdout"
)

shuf_summary, _ = evaluate(
    shuffled,
    score_col="zero_shot_adapter_score",
    model="ZeroShotAdapter",
    split="holdout"
)

check_cols = ["ndcg@10", "ndcg@3", "recall@10", "recall@3", "mrr"]
max_diff = max(abs(orig_summary[c] - shuf_summary[c]) for c in check_cols)

print("Maximum metric difference after row shuffle:", max_diff)
assert max_diff < 1e-12, "ERROR: ranking still depends on input row order."
print("PASS: evaluation is independent of input row order.")


In [ ]:

# 11. PLOTS

for metric, filename, title in [
    ("ndcg@10", "v2_fixed_holdout_ndcg10.png", "Week 6 V2 FIXED — Held-Out NDCG@10"),
    ("ndcg@3", "v2_fixed_holdout_ndcg3.png", "Week 6 V2 FIXED — Held-Out NDCG@3"),
    ("mrr", "v2_fixed_holdout_mrr.png", "Week 6 V2 FIXED — Held-Out MRR"),
]:
    p = holdout_metrics.dropna(subset=[metric]).sort_values(metric, ascending=False)
    plt.figure(figsize=(9, 5))
    plt.bar(p["model"], p[metric])
    plt.ylabel(metric.upper())
    plt.xlabel("Model")
    plt.title(title)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()

weights_df = pd.DataFrame({
    "feature_group": groups,
    "source_column": [selected[g] for g in groups],
    "weight": best_w,
}).sort_values("weight", ascending=False)

plt.figure(figsize=(9, 5))
plt.bar(weights_df["feature_group"], weights_df["weight"])
plt.ylabel("Weight")
plt.xlabel("Zero-shot feature")
plt.title("Week 6 V2 FIXED — Adapter Weights")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "v2_fixed_weights.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

# 12. SAVE RESULTS

weights_df.to_csv(OUT_DIR / "zero_shot_best_weights.csv", index=False)
feature_diag.to_csv(OUT_DIR / "zero_shot_feature_diagnostics.csv", index=False)
search_results.head(200).to_csv(
    OUT_DIR / "zero_shot_weight_search_top200.csv", index=False
)
holdout_metrics.to_csv(
    OUT_DIR / "zero_shot_holdout_metrics.csv", index=False
)
full_metrics.to_csv(
    OUT_DIR / "zero_shot_full_coldstart_metrics.csv", index=False
)
per_query.to_csv(
    OUT_DIR / "zero_shot_per_query_metrics.csv", index=False
)

scored_cols = [
    c for c in [
        "__query__", "__product__", "query", "title", "__label__",
        "v2_split", "zero_shot_adapter_score",
        "bm25_score_qnorm", "sbert_similarity_qnorm",
        "clip_similarity_qnorm", "lambdamart_score", "listnet_score"
    ] if c in cold.columns
]
cold[scored_cols].to_csv(
    OUT_DIR / "zero_shot_scored_pairs.csv", index=False
)

policy = {
    "prohibited_keywords": PROHIBITED_KEYWORDS,
    "detected_features": dict(detected),
    "selected_features": dict(selected),
    "min_varying_query_share": MIN_VARYING_QUERY_SHARE,
    "tie_breaker": "SHA1(product_id) deterministic hash",
    "row_order_independence_test_passed": True,
}
with open(OUT_DIR / "zero_shot_feature_policy.json", "w") as f:
    json.dump(policy, f, indent=2)

z = holdout_metrics[
    holdout_metrics["model"] == "ZeroShotAdapter"
].iloc[0]

lines = [
    "Amazon Search Ranking — Week 6 V2 FIXED",
    "Leakage-Safe Zero-Shot Ranking Adapter",
    "=" * 64,
    "",
    "Important fix:",
    "- Ties use a deterministic product hash, not original file row order.",
    "- Low-information zero-shot features are removed automatically.",
    "- Row-shuffle invariance test passed.",
    "",
    "Held-out Zero-Shot Adapter:",
    f"- NDCG@10: {z['ndcg@10']:.4f}",
    f"- NDCG@3: {z['ndcg@3']:.4f}",
    f"- Recall@3: {z['recall@3']:.4f}",
    f"- MRR: {z['mrr']:.4f}",
    f"- NDCG@10 vs BM25: {z['ndcg10_improvement_vs_bm25_pct']:+.2f}%",
    f"- 15% target achieved: {'YES' if z['ndcg10_improvement_vs_bm25_pct'] >= 15 else 'NO'}",
    "",
    "Best adapter weights:"
]

for _, r in weights_df.iterrows():
    lines.append(
        f"- {r['feature_group']}: {r['weight']:.4f} ({r['source_column']})"
    )

summary_text = "\n".join(lines)
(OUT_DIR / "week6_v2_fixed_summary.txt").write_text(summary_text)

print(summary_text)


In [ ]:

# 13. PACKAGE + DOWNLOAD

zip_path = ROOT / "amazon_week6_v2_fixed_zero_shot_adapter_results.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT_DIR))

print("Created:", zip_path)

from google.colab import files
files.download(str(zip_path))



## Decision after V2 FIXED

Use the **held-out** results as the official V2 result.

If the Zero-Shot Adapter does not reach the +15% NDCG@10 target, that is a valid experimental outcome rather than a failure of the project.

It motivates Week 6 V3:

**LLM-generated relevance signal → add semantic reasoning that is available even when the product has little/no behavioral history.**
